# Week 3 Assignment — Financial Features + Agent Design

**Multi-Agent Financial Forecasting · Stamatics IIT Kanpur**  
**Mentors:** Aayushman Tripathi, Manthan Khetade, Arisht Daiya

---

## What You'll Build This Week

In Weeks 1–2 you learned the mechanics of time series and ML models. This week you connect those tools to **real financial data** and start building the actual agents for our multi-agent pipeline.

By the end of this notebook you will have:
1. Computed log returns, RSI, MACD, and Bollinger Band Width from scratch on real SPY data
2. Understood *why* each feature contains predictive signal
3. Written a proper `BaseAgent` abstract class and two concrete agents
4. Understood the Hedge algorithm — the theoretical heart of the project

---

## Prerequisites

```
pip install yfinance pandas numpy matplotlib scikit-learn xgboost statsmodels
```

## Part 1 — Log Returns: The Right Prediction Target

### Why Not Prices?

SPY (S&P 500 ETF) closed at about **\$115 in January 2010** and **\$480 in December 2024**. If you train a model on 2010–2015 price data and test it on 2024, the model has **never seen inputs in the $400–500 range**. The statistical distribution of the inputs has completely shifted.

This is called **non-stationarity**: the statistical properties of the series change over time.

**Log returns** fix this:
$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns are:
- **Approximately stationary**: ~1% daily std, mean near 0, regardless of the price level
- **Additive**: the k-period log return is exactly $r_1 + r_2 + ... + r_k$
- **Closer to Gaussian**: ML models work better with near-normal inputs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# Download SPY data
spy = yf.download('SPY', start='2018-01-01', end='2023-12-31', progress=False)
spy = spy['Close'].squeeze()  # Series of closing prices

print(f'Price range: ${spy.min():.2f} to ${spy.max():.2f}')
print(f'Sample prices (first 5):')
print(spy.head())

In [ ]:
# Compute log returns
log_returns = np.log(spy / spy.shift(1)).dropna()

print(f'Log return stats:')
print(f'  Mean: {log_returns.mean():.4f} ({log_returns.mean()*252:.2f}% annualized)')
print(f'  Std:  {log_returns.std():.4f} ({log_returns.std()*np.sqrt(252):.2%} annualized vol)')
print(f'  Min:  {log_returns.min():.4f} (worst day)')
print(f'  Max:  {log_returns.max():.4f} (best day)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
spy.plot(ax=axes[0], title='SPY Price (non-stationary)', color='steelblue')
axes[0].set_ylabel('Price ($)')
log_returns.plot(ax=axes[1], title='SPY Log Returns (approximately stationary)',
                  color='darkorange', alpha=0.7)
axes[1].set_ylabel('Log Return')
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

### Exercise 1.1
**What was the worst single day for SPY in your data? What date was it, and what was the log return?**

*(Hint: use `.idxmin()` on the log_returns Series)*

In [ ]:
# Your code here
worst_day = log_returns.idxmin()
worst_return = log_returns[worst_day]
print(f'Worst day: {worst_day.date()}, log return: {worst_return:.4f} ({(np.exp(worst_return)-1)*100:.2f}%)')

---

## Part 2 — The Lookahead Bias Problem

This is the **#1 failure mode** in financial machine learning. Every feature you compute must use **only information available before the trading day you're predicting**.

The fix is always `.shift(1)`: shift features forward by one day so that the feature for day $t$ uses data from day $t-1$ and earlier.

```python
# WRONG — uses today's close to predict today's return
rsi = compute_rsi(close)          # close[t] is today's price

# CORRECT — uses yesterday's close (and earlier) to predict today's return
rsi = compute_rsi(close.shift(1)) # close.shift(1)[t] = close[t-1]
```

Without this shift, your backtest will look spectacular — because the model is "seeing" the answer before predicting it. When deployed live, performance collapses.

In [ ]:
# Demonstrate the impact of lookahead bias
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

close = spy.copy()
target = log_returns.copy()

# Feature: yesterday's return as predictor
lag1_correct = log_returns.shift(1)          # Uses t-1 to predict t — correct
lag1_wrong   = log_returns                   # Uses t to predict t — WRONG (lookahead!)

df = pd.DataFrame({'target': target, 'correct': lag1_correct, 'wrong': lag1_wrong}).dropna()

# Split: train on 2018-2021, test on 2022-2023
train = df[df.index.year <= 2021]
test  = df[df.index.year >= 2022]

for name, feat in [('correct (no lookahead)', 'correct'), ('WRONG (lookahead!)', 'wrong')]:
    model = LinearRegression()
    model.fit(train[[feat]], train['target'])
    preds = model.predict(test[[feat]])
    mae = mean_absolute_error(test['target'], preds)
    da = np.mean(np.sign(preds) == np.sign(test['target']))
    print(f'{name:35s} — MAE: {mae:.6f} | Dir. Accuracy: {da:.1%}')

---

## Part 3 — Technical Indicators from Scratch

### 3.1 RSI — Relative Strength Index

RSI measures whether a stock has been going up or down more on recent days. It oscillates between 0 and 100.

- **RSI > 70**: overbought — may revert down
- **RSI < 30**: oversold — may revert up

**Formula:**
$$\text{RSI} = 100 - \frac{100}{1 + RS}, \quad RS = \frac{\text{EWM average of gains}}{\text{EWM average of losses}}$$

We use EWM (Exponentially Weighted Moving Average) instead of simple averages because:
1. Recent days matter more than 14 days ago
2. SMA would have discontinuous jumps when old extreme values leave the window

In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """RSI using EWM smoothing. com=period-1 gives alpha=1/period."""
    delta = series.diff()
    gains = delta.clip(lower=0)   # zero out negative days
    losses = -delta.clip(upper=0) # zero out positive days (flip sign)
    
    # EWM with com=period-1 → alpha = 1/(1+com) = 1/period
    avg_gain = gains.ewm(com=period - 1, adjust=False).mean()
    avg_loss = losses.ewm(com=period - 1, adjust=False).mean()
    
    rs = avg_gain / (avg_loss + 1e-10)  # 1e-10 to avoid division by zero
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Compute RSI on shifted close (no lookahead)
rsi = compute_rsi(close.shift(1).dropna())

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
close.plot(ax=axes[0], title='SPY Price', color='steelblue')
rsi.plot(ax=axes[1], title='RSI-14', color='darkorange')
axes[1].axhline(70, color='red', linestyle='--', linewidth=0.8, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
axes[1].axhline(50, color='gray', linestyle=':', linewidth=0.5)
axes[1].set_ylim(0, 100)
axes[1].legend()
plt.tight_layout()
plt.show()

### 3.2 MACD — Moving Average Convergence Divergence

MACD captures the momentum of a stock by looking at the gap between a fast and slow moving average.

- **Fast EMA (12 days)**: responds quickly to recent price changes
- **Slow EMA (26 days)**: tracks longer-term trend
- **MACD line**: fast − slow (positive = short-term momentum above long-term trend)
- **Signal line**: 9-day EMA of MACD (smoothed version — **this is what we use as a feature**)

In [ ]:
def compute_macd(series: pd.Series,
                 fast: int = 12, slow: int = 26, signal: int = 9) -> pd.DataFrame:
    """Returns DataFrame with 'macd_line' and 'macd_signal' columns."""
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    macd_signal = macd_line.ewm(span=signal, adjust=False).mean()
    return pd.DataFrame({'macd_line': macd_line, 'macd_signal': macd_signal})

macd = compute_macd(close.shift(1).dropna())

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
close.plot(ax=axes[0], title='SPY Price', color='steelblue')
macd['macd_line'].plot(ax=axes[1], title='MACD', color='navy', label='MACD line')
macd['macd_signal'].plot(ax=axes[1], color='darkorange', label='Signal line')
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].legend()
plt.tight_layout()
plt.show()

### 3.3 Bollinger Band Width

Bollinger Bands are a volatility envelope around a 20-day simple moving average:
- **Upper band**: SMA + 2 × std
- **Lower band**: SMA − 2 × std
- **Width** = (upper − lower) / mid = **4 × std / SMA**

Width is a normalized volatility measure. When width is low (narrow bands = quiet market), a big move is often coming. This is the "Bollinger Squeeze."

**Why SMA, not EWM here?** Bollinger designed the bands around a stable, symmetric baseline. EWM would make the center line responsive to recent moves, which defeats the purpose — you want the band to be a stable reference, not chasing the price.

In [ ]:
def compute_bollinger(series: pd.Series, window: int = 20) -> pd.Series:
    """Returns Bollinger Band Width = (upper - lower) / mid = 4 * std / SMA."""
    sma = series.rolling(window).mean()
    std = series.rolling(window).std()
    upper = sma + 2 * std
    lower = sma - 2 * std
    width = (upper - lower) / sma  # normalized width
    return width

bb_width = compute_bollinger(close.shift(1).dropna())

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Price with bands
shifted_close = close.shift(1).dropna()
sma20 = shifted_close.rolling(20).mean()
std20 = shifted_close.rolling(20).std()
axes[0].plot(close.index, close, color='steelblue', linewidth=1, label='Price')
axes[0].plot(sma20.index, sma20 + 2*std20, color='gray', linewidth=0.7, linestyle='--', label='Upper band')
axes[0].plot(sma20.index, sma20 - 2*std20, color='gray', linewidth=0.7, linestyle='--', label='Lower band')
axes[0].fill_between(sma20.index, sma20+2*std20, sma20-2*std20, alpha=0.1, color='gray')
axes[0].set_title('SPY with Bollinger Bands (20d)')
axes[0].legend()

bb_width.plot(ax=axes[1], title='Bollinger Band Width (normalized vol)', color='purple')
axes[1].axhline(bb_width.mean(), color='gray', linestyle='--', linewidth=0.8,
                label=f'Mean ({bb_width.mean():.3f})')
axes[1].legend()
plt.tight_layout()
plt.show()

### Exercise 3.1
**Find the date when Bollinger Band Width was at its minimum (narrowest bands). What happened to SPY in the days immediately after?**

This tests whether you see the "Bollinger Squeeze" (low volatility often precedes a big move) in real data.

In [ ]:
# Your code here
squeeze_date = bb_width.idxmin()
print(f'Narrowest bands: {squeeze_date.date()}, width={bb_width[squeeze_date]:.4f}')

# Look at price action in the 10 days after
window_after = log_returns.loc[squeeze_date:].head(10)
print(f'\nLog returns in next 10 days:')
print(window_after)

---

## Part 4 — Building the Feature Matrix

Now combine everything into a clean feature matrix. **Every column uses `.shift(1)` on the close price** — the single most important line in the pipeline.

The output: a DataFrame where each row is one trading day. Features are computed from data available *before* that day. The target column (`log_return`) is what happened *on* that day.

In [ ]:
def build_feature_matrix(spy_close: pd.Series, log_ret: pd.Series,
                         vix: pd.Series | None = None) -> pd.DataFrame:
    """
    Build a lookahead-free feature matrix.
    All price-based features use .shift(1).
    Target: log_return (today's, what we want to predict).
    """
    # Keystone: shift close prices so we only see yesterday's data
    c = spy_close.shift(1)   # c[t] = close[t-1]
    r = log_ret.shift(1)     # r[t] = return[t-1]
    
    features = pd.DataFrame(index=spy_close.index)
    
    # Lag features
    for lag in [1, 2, 3, 5, 10]:
        features[f'lag_{lag}'] = log_ret.shift(lag)
    
    # Rolling statistics on shifted returns
    for window in [5, 21]:
        features[f'rolling_mean_{window}'] = r.rolling(window).mean()
        features[f'rolling_std_{window}']  = r.rolling(window).std()
    
    # Technical indicators on shifted close
    features['rsi_14']      = compute_rsi(c)
    features['macd_signal'] = compute_macd(c)['macd_signal']
    features['bb_width']    = compute_bollinger(c)
    
    # VIX features (if available)
    if vix is not None:
        vix_shifted = vix.shift(1)
        features['vix_level']    = vix_shifted
        features['vix_change_5d'] = vix_shifted.diff(5)
    
    # Target: today's log return (what we're predicting)
    features['log_return'] = log_ret
    
    return features.dropna()

# Build it
feat_df = build_feature_matrix(close, log_returns)
print(f'Feature matrix: {feat_df.shape[0]} rows × {feat_df.shape[1]} columns')
print(f'\nColumns: {list(feat_df.columns)}')
print(f'\nFirst row (day 1 of out-of-sample data):')
print(feat_df.iloc[0])

---

## Part 5 — The Agent Design Pattern

All four agents in our pipeline share the same interface: `.fit(train_df)` and `.predict(test_df)`. This uniformity is what lets the Hedge algorithm treat them interchangeably.

We enforce this with an **Abstract Base Class (ABC)** — a class that defines the interface but doesn't implement it. Any subclass that forgets to implement `fit` or `predict` will raise an error immediately.

In [ ]:
from abc import ABC, abstractmethod

class BaseAgent(ABC):
    """Contract: every agent must implement fit() and predict()."""
    
    def __init__(self, name: str):
        self.name = name
    
    @abstractmethod
    def fit(self, train_df: pd.DataFrame) -> None:
        """Train on historical data."""
        ...
    
    @abstractmethod
    def predict(self, test_df: pd.DataFrame) -> np.ndarray:
        """Return predicted log returns for each row in test_df."""
        ...


# Try to instantiate BaseAgent directly — Python will refuse
try:
    agent = BaseAgent('test')
except TypeError as e:
    print(f'Error (expected): {e}')
    print('→ ABCs cannot be instantiated. You must subclass them and implement all @abstractmethods.')

### 5.1 Momentum Agent (XGBoost)

The MomentumAgent uses XGBoost on lag features and rolling statistics. It captures short-term serial dependence: the idea that recent up-days or down-days contain information about tomorrow.

In [ ]:
from xgboost import XGBRegressor

MOMENTUM_FEATURES = [
    'lag_1', 'lag_2', 'lag_3', 'lag_5', 'lag_10',
    'rolling_mean_5', 'rolling_mean_21', 'rolling_std_5',
    'rsi_14', 'macd_signal'
]

class MomentumAgent(BaseAgent):
    """XGBoost on momentum + technical features."""
    
    def __init__(self):
        super().__init__('MomentumAgent')
        self._model = XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            random_state=42,
            verbosity=0,
        )
    
    def fit(self, train_df: pd.DataFrame) -> None:
        avail = [f for f in MOMENTUM_FEATURES if f in train_df.columns]
        X = train_df[avail]
        y = train_df['log_return']
        self._model.fit(X, y)
        self._features = avail
    
    def predict(self, test_df: pd.DataFrame) -> np.ndarray:
        return self._model.predict(test_df[self._features])
    
    def feature_importance(self) -> pd.Series:
        return pd.Series(
            self._model.feature_importances_,
            index=self._features
        ).sort_values(ascending=False)


# Quick test
train = feat_df.iloc[:1000]
test  = feat_df.iloc[1000:1252]

momentum = MomentumAgent()
momentum.fit(train)
preds = momentum.predict(test)

mae = np.mean(np.abs(preds - test['log_return'].values))
da  = np.mean(np.sign(preds) == np.sign(test['log_return'].values))
print(f'MomentumAgent — MAE: {mae:.6f} | Directional Accuracy: {da:.1%}')

print('\nTop 5 feature importances:')
print(momentum.feature_importance().head())

### Exercise 5.1
**Write a `VolatilityAgent` that uses XGBoost on the Bollinger width and rolling standard deviation features.**

Features to use: `['bb_width', 'rolling_std_5', 'rolling_std_21']`

If you have VIX in your feature matrix, also add `'vix_level'` and `'vix_change_5d'`.

In [ ]:
VOLATILITY_FEATURES = ['bb_width', 'rolling_std_5', 'rolling_std_21']

class VolatilityAgent(BaseAgent):
    """Fill in: XGBoost on volatility features."""
    
    def __init__(self):
        super().__init__('VolatilityAgent')
        # YOUR CODE: create an XGBRegressor
        ...
    
    def fit(self, train_df: pd.DataFrame) -> None:
        # YOUR CODE: fit the model on VOLATILITY_FEATURES
        ...
    
    def predict(self, test_df: pd.DataFrame) -> np.ndarray:
        # YOUR CODE: return predictions
        ...

# Test your implementation
vol_agent = VolatilityAgent()
vol_agent.fit(train)
vol_preds = vol_agent.predict(test)
vol_da = np.mean(np.sign(vol_preds) == np.sign(test['log_return'].values))
print(f'VolatilityAgent — Directional Accuracy: {vol_da:.1%}')

---

## Part 6 — The Hedge Algorithm

You now have multiple agents. How do you combine their predictions?

**Option 1: Simple average.** Take `(pred_1 + pred_2 + ... + pred_N) / N`. This is unweighted — every agent counts equally regardless of recent performance.

**Option 2: Hedge algorithm.** Maintain a weight vector. After each prediction, reduce the weight of agents that made large errors. This is **online learning** — the ensemble adapts to which agents are currently most reliable.

### The Update Rule

$$w_i \leftarrow w_i \cdot e^{-\eta \cdot L_i}$$

Then normalize: $w_i \leftarrow w_i / \sum_j w_j$

Where:
- $w_i$ = weight of agent $i$
- $\eta$ (eta) = learning rate — controls how aggressively to downweight bad agents
- $L_i$ = squared error of agent $i$ today: $(\hat{y}_i - y)^2$

### Why Exponential?

Exponential decay has a nice property: no weight ever reaches exactly zero. Every agent always has *some* chance to contribute. This is important because a bad agent in a bull market might be exactly the right agent in a bear market — the exponential update keeps it in the mix.

In [ ]:
class HedgeAggregator:
    """
    Freund & Schapire (1997) Hedge algorithm.
    
    Regret bound: R_T ≤ sqrt(T * log(N) / 2) for eta = sqrt(log(N) / (2T))
    Guarantee: ensemble performance ≥ best single agent, minus a vanishing term.
    """
    
    def __init__(self, n_agents: int, eta: float = 0.1):
        self.n_agents = n_agents
        self.eta = eta
        self.weights = np.ones(n_agents) / n_agents  # start uniform
        self.weight_history = []
    
    def aggregate(self, predictions: list[float]) -> float:
        """Weighted average of agent predictions."""
        return float(np.dot(self.weights, predictions))
    
    def update(self, predictions: list[float], actual: float) -> None:
        """Multiplicative weights update after observing actual value."""
        losses = (np.array(predictions) - actual) ** 2
        self.weights *= np.exp(-self.eta * losses)
        self.weights /= self.weights.sum()  # normalize to sum=1
        self.weight_history.append(self.weights.copy())
    
    def reset(self):
        """Reset for a new backtest run."""
        self.weights = np.ones(self.n_agents) / self.n_agents
        self.weight_history.clear()
    
    @staticmethod
    def optimal_eta(T: int, N: int) -> float:
        """Theoretically optimal learning rate."""
        return np.sqrt(np.log(N) / (2 * T))


# Simulate Hedge vs simple average on our test set
agents_list = [MomentumAgent()]

# Add the VolatilityAgent if you implemented it above:
# agents_list.append(VolatilityAgent())

# Fit all agents
for a in agents_list:
    a.fit(train)

hedge = HedgeAggregator(n_agents=len(agents_list), eta=0.1)

actuals = test['log_return'].values
hedge_preds = []
equal_preds = []

# Simulate day by day (online learning)
all_agent_preds = [a.predict(test) for a in agents_list]
for i in range(len(actuals)):
    step_preds = [float(p[i]) for p in all_agent_preds]
    
    # Hedge prediction
    hedge_preds.append(hedge.aggregate(step_preds))
    
    # Simple average
    equal_preds.append(float(np.mean(step_preds)))
    
    # Update weights
    hedge.update(step_preds, actuals[i])

hedge_preds = np.array(hedge_preds)
equal_preds = np.array(equal_preds)

print('Performance comparison:')
for name, preds in [('Hedge', hedge_preds), ('Equal Weight', equal_preds)]:
    da = np.mean(np.sign(preds) == np.sign(actuals))
    mae = np.mean(np.abs(preds - actuals))
    print(f'  {name:15s} — Dir. Acc: {da:.1%} | MAE: {mae:.6f}')

In [ ]:
# Plot weight evolution (if you have 2+ agents)
if len(hedge.weight_history) > 0 and len(agents_list) > 1:
    weight_df = pd.DataFrame(hedge.weight_history,
                              columns=[a.name for a in agents_list],
                              index=test.index)
    
    fig, ax = plt.subplots(figsize=(14, 4))
    weight_df.plot(ax=ax, title='Hedge Weight Evolution')
    ax.set_ylabel('Weight')
    ax.set_ylim(0, 1)
    ax.axhline(1/len(agents_list), color='gray', linestyle='--',
               linewidth=0.8, label='Equal weight')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Add your VolatilityAgent above to see the weight evolution chart!')

### Exercise 6.1 — η Sensitivity

The learning rate η controls how fast the Hedge algorithm reacts to agent performance differences.

**High η**: fast reaction — if an agent has two bad days, its weight drops sharply. Risky: might abandon a good agent temporarily.

**Low η**: slow reaction — weights change gradually, close to equal weighting. Safer but slower to adapt.

**Optimal η** = √(log(N) / (2T))

For N=2 agents, T=252 days (1 year): η_opt ≈ ?

In [ ]:
# Compute optimal eta for different horizons
N = 4  # 4 agents in full pipeline
for T in [252, 504, 1008, 3520]:
    eta_opt = HedgeAggregator.optimal_eta(T, N)
    print(f'T={T:5d} days, N={N} agents → η_opt = {eta_opt:.5f}')

# Run Hedge with different eta values and compare
print('\nDirectional accuracy at different η values:')
for eta in [0.001, 0.01, 0.1, 1.0, 10.0]:
    h = HedgeAggregator(n_agents=len(agents_list), eta=eta)
    preds_eta = []
    for i in range(len(actuals)):
        step_preds = [float(p[i]) for p in all_agent_preds]
        preds_eta.append(h.aggregate(step_preds))
        h.update(step_preds, actuals[i])
    da = np.mean(np.sign(np.array(preds_eta)) == np.sign(actuals))
    print(f'  η={eta:6.3f} → Dir. Acc: {da:.1%}')

---

## Part 7 — Putting It Together

### Theoretical Guarantee

The Hedge algorithm has a provable regret bound:

$$R_T = \sum_{t=1}^{T} L^{\text{ensemble}}_t - \min_i \sum_{t=1}^{T} L^i_t \leq \sqrt{\frac{T \cdot \log N}{2}}$$

This says: the *total* loss of the Hedge ensemble over T rounds cannot exceed the total loss of the **best individual agent** by more than $\sqrt{T \log N / 2}$. Per step, the excess is at most $\sqrt{\log N / (2T)}$ → 0 as T → ∞.

In plain language: **no matter which agent turns out to be best in hindsight, Hedge will eventually perform as well as that agent.** No agent knowledge is wasted.

### System Summary Table

In [ ]:
summary = pd.DataFrame({
    'Agent': ['TrendAgent', 'MomentumAgent', 'VolatilityAgent', 'SequenceAgent'],
    'Model': ['LinearRegression + DeterministicProcess', 'XGBoost', 'XGBoost', 'LSTM (PyTorch)'],
    'Key Features': [
        'Time index, Fourier terms',
        'lag_1..lag_10, RSI, MACD',
        'bb_width, rolling_std, VIX',
        'Raw return sequences (window=30)'
    ],
    'Captures': [
        'Long-run trend + seasonality',
        'Short-term momentum',
        'Volatility regime shifts',
        'Non-linear temporal patterns'
    ]
})
print(summary.to_string(index=False))

---

## Final Exercises

### Exercise 7.1 — Implement a Naive Baseline
Write an agent that always predicts `0.0` (flat — no position). What is its directional accuracy? What does this tell you about the difficulty of the problem?

### Exercise 7.2 — Implement a Yesterday's Return Agent
Write an agent that predicts `lag_1` as its forecast ("tomorrow will repeat today"). Compare its MAE and directional accuracy to MomentumAgent. Is momentum a real signal or just noise?

### Exercise 7.3 — RSI Contrarian Signal
The classic RSI strategy: predict negative return when RSI > 70 (overbought) and positive return when RSI < 30 (oversold). Implement this rule-based agent and evaluate its directional accuracy on the test set.

*(Hint: no ML model needed — just a simple if/else on the RSI column)*

In [ ]:
# Exercise 7.1 — Naive baseline
class NaiveAgent(BaseAgent):
    """Always predicts 0 — no position."""
    def __init__(self): super().__init__('NaiveAgent')
    def fit(self, train_df): pass
    def predict(self, test_df): return np.zeros(len(test_df))

naive = NaiveAgent()
naive.fit(train)
naive_preds = naive.predict(test)
naive_da = np.mean(np.sign(naive_preds) == np.sign(actuals))
print(f'NaiveAgent (always 0) — Directional Accuracy: {naive_da:.1%}')
print('(This should be ~50% since sign(0) matches sign(actual) only when actual=0)')

# Exercise 7.2 — YOUR CODE

# Exercise 7.3 — YOUR CODE

---

## Summary

| Concept | Key Formula / Code | Why It Matters |
|---|---|---|
| Log returns | `np.log(P_t / P_{t-1})` | Stationary, additive, near-Gaussian |
| Lookahead prevention | `.shift(1)` on all price-based features | #1 failure mode in financial ML |
| RSI | EWM gains / EWM losses, scaled 0–100 | Overbought / oversold signal |
| MACD signal | 9-EMA of (fast-EMA − slow-EMA) | Smoothed momentum signal |
| Bollinger width | `4 × std / SMA` | Normalized volatility; squeeze precedes moves |
| ABC | `@abstractmethod` on `fit`, `predict` | Uniform interface for all agents |
| Hedge update | `w_i *= exp(−η × L_i)`, then normalize | Online learning; adapts to agent performance |
| Regret bound | `O(sqrt(T × log N))` | Hedge ≥ best agent in hindsight, eventually |

**Next week (Week 4):** Walk-forward backtesting — putting all agents and the Hedge aggregator together into a rigorous evaluation framework that avoids temporal leakage.

---

*Multi-Agent Financial Forecasting · Stamatics IIT Kanpur · 2026*